In [2]:
import pandas as pd
import sklearn

df = pd.read_csv("../data/processed/BRinf_clean.csv", index_col=0, parse_dates=True)

df.head()

,ipca_mom,igpm_mom,igpdi_mom,fgv_igp_unknown,igp10_mom,focus_ipca_12m_median,electricity_consumption_total,electricity_consumption_ind,bofa_economic_condition_idx,cni_capacity_util_diff,...,focus_top5_med_t12,focus_top5_med_t13,focus_med_t1_sq,focus_mean_t1,focus_mean_t1_sq,focus_std_t1,focus_med_t2_sq,focus_mean_t2,focus_mean_t2_sq,focus_std_t2
2003-03-31,1.23,1.53,1.66,4.593,1.58,9.58,24960.0,115.0,-0.35,-0.3,...,0.55,0.50,0.8836,0.93,0.8649,0.13,0.4900,0.73,0.5329,0.17
2003-04-30,0.97,0.92,0.41,1.173,1.24,8.88,25284.0,93.0,-0.15,-0.5,...,0.45,0.78,1.0000,0.97,0.9409,0.14,0.2500,0.52,0.2704,0.18
2003-05-31,0.61,-0.26,-0.67,-1.860,0.02,8.22,24785.0,285.0,-0.47,0.0,...,0.60,0.43,0.2500,0.50,0.2500,0.10,0.3025,0.55,0.3025,0.15
2003-06-30,-0.15,-1.00,-0.70,-1.949,-0.59,6.91,24889.0,-99.0,-0.73,0.1,...,0.51,0.55,0.0100,0.17,0.0289,0.17,1.1025,1.07,1.1449,0.21
2003-07-31,0.20,-0.42,-0.20,-0.544,-0.73,6.29,24536.0,38.0,-0.14,-0.5,...,0.50,0.97,0.0900,0.39,0.1521,0.28,0.3364,0.56,0.3136,0.18


In [3]:
from ipca_forecast.features import train_preparation

X_train, y_train, X_test, y_test = train_preparation(df, 3)

In [9]:
top25 = X_train.corrwith(y_train).sort_values(key=abs, ascending=False).iloc[:25]

top25

treasury_rev_import_tax      0.310674
unemployment_rate_imf       -0.302177
cni_capacity_utilization     0.294701
savings_deposits_total       0.268672
focus_top5_med_t3            0.230324
focus_med_t3                 0.229424
focus_med_t2                 0.224206
focus_top5_med_t9           -0.222797
focus_mean_t2                0.218953
focus_top5_med_t4            0.215924
pis_pasep_tax                0.213049
focus_med_t2_sq              0.212119
focus_top5_med_t6            0.202569
focus_mean_t2_sq             0.199633
focus_top5_med_t2            0.187702
focus_med_t4                 0.180637
trade_balance_fob            0.171861
focus_top5_med_t7            0.167813
central_gov_net_revenue      0.159020
unemployment_rate_ibge      -0.158992
social_contrib_net_profit    0.158725
treasury_gross_revenue       0.152416
unemployment_male           -0.151001
focus_med_t6                 0.146971
treasury_revenue_total       0.143341
dtype: float64

In [15]:
import itertools

groups = list(itertools.combinations(top25.index, 4))
groups[:4]

[('treasury_rev_import_tax',
  'unemployment_rate_imf',
  'cni_capacity_utilization',
  'savings_deposits_total'),
 ('treasury_rev_import_tax',
  'unemployment_rate_imf',
  'cni_capacity_utilization',
  'focus_top5_med_t3'),
 ('treasury_rev_import_tax',
  'unemployment_rate_imf',
  'cni_capacity_utilization',
  'focus_med_t3'),
 ('treasury_rev_import_tax',
  'unemployment_rate_imf',
  'cni_capacity_utilization',
  'focus_med_t2')]

In [21]:
from sklearn.linear_model import LinearRegression

predictions = []

for c in groups:
    cols = list(c)

    model = LinearRegression()
    model.fit(X_train[cols], y_train)
    pred = model.predict(X_test[cols])

    predictions.append(pred)




In [ ]:
import numpy as np

prediction_array = np.array(predictions)

csr_predictions = prediction_array.mean(axis=0)

In [25]:
mae_csr = sklearn.metrics.mean_absolute_error(y_test, csr_predictions)

rmse_csr = sklearn.metrics.root_mean_squared_error(y_test, csr_predictions)

print(f"mae_csr = {mae_csr}\nrmse_csr = {rmse_csr}")

mae_csr = 0.20729649596880256
rmse_csr = 0.2708298617816253
